# Local Databricks Lakehouse: SQLFrame + DuckDB + duckrun

This notebook demonstrates full local PySpark parity without a JVM or cloud cluster:
- **PySpark DataFrame API**: Transpiled via **SQLFrame** directly to DuckDB.
- **Compute Engine**: Vectorized C++ execution via **DuckDB**.
- **ACID Lakehouse Storage**: Full Delta Lake commits, ACID transactions, and time travel via **duckrun** and **delta-rs**.
- **Databricks UX**: Global `spark`, `dbutils`, `display()`, and `%sql` / `%%sql` preloaded.

## Step 1: PySpark DataFrame Operations via SQLFrame

In [1]:
from sqlframe.duckdb import functions as F
from sqlframe.duckdb.window import Window

data = [
    ("Alice", "Engineering", 125000, "2024-01-15"),
    ("Bob", "Engineering", 95000, "2024-02-10"),
    ("Charlie", "Finance", 110000, "2024-03-01"),
    ("David", "Finance", 85000, "2024-01-20"),
    ("Eve", "Engineering", 130000, "2024-04-12"),
]
columns = ["name", "department", "salary", "hire_date"]

df = spark.createDataFrame(data, schema=columns)

# Standard window operations
window_spec = Window.partitionBy("department").orderBy(F.col("salary").desc())

transformed_df = (
    df.withColumn("rank", F.dense_rank().over(window_spec))
      .withColumn("bonus_estimate", F.round(F.col("salary") * 0.12, 2))
)

# Interactive Databricks display table
display(transformed_df)

/usr/local/lib/python3.11/site-packages/sqlframe/base/session.py:588: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return read_sql_query(


## Step 2: ACID Delta Lake Persistence via duckrun
We materialize the DataFrame directly into our local Delta Lake warehouse storage as an ACID Delta table.

In [ ]:
# Materialize into the Delta table warehouse using duckrun
conn.sql("""
    CREATE OR REPLACE TABLE silver_employees AS 
    SELECT * FROM transformed_df
""")
print("Delta table silver_employees materialized successfully.")

## Step 3: Inspect Stored Delta Directory using dbutils

In [2]:
# Inspect stored directory using dbutils mock
files = dbutils.fs.ls("dbfs:/silver_employees")
for f in files:
    print(f"{f['name']:<50} {f['size']:>8} bytes  (Dir: {f['isDir']})")

_delta_log                                             4096 bytes  (Dir: True)
part-00000-7207e871-5e6e-4386-82e2-fd2a0e1caa4b-c000.snappy.parquet     2149 bytes  (Dir: False)
part-00000-8276649d-181f-4974-97cc-04083b430a12-c000.snappy.parquet     1913 bytes  (Dir: False)
part-00000-069ba22f-457a-41ba-8e06-279c4535ea90-c000.snappy.parquet     1910 bytes  (Dir: False)
part-00000-bb3db007-1ef7-4bcf-916b-4d0516b6e97a-c000.snappy.parquet     1020 bytes  (Dir: False)
part-00000-2b04ef1d-896c-455f-9477-9c9a10527b1e-c000.snappy.parquet     1913 bytes  (Dir: False)
part-00000-f9e15372-1e34-4c8e-82dc-728bb3afc3f0-c000.snappy.parquet      894 bytes  (Dir: False)
part-00000-68d8f0ba-bacb-4c11-ac99-2a4d21e01cef-c000.snappy.parquet     2149 bytes  (Dir: False)


## Step 4: Direct Delta Table Querying via %%sql Magic

In [2]:
%%sql
SELECT 
    department,
    COUNT(*) AS total_employees,
    ROUND(AVG(salary), 2) AS avg_salary,
    SUM(bonus_estimate) AS total_bonus
FROM silver_employees
GROUP BY department
ORDER BY avg_salary DESC;

## Step 5: ACID Mutations & Time Travel
Demonstrate full ACID transactions and Delta Lake time travel capabilities.

In [ ]:
# ACID Update statement
conn.sql("UPDATE silver_employees SET salary = salary + 5000 WHERE name = 'Alice'")

print("Updated current state:")
display(conn.sql("SELECT name, salary FROM silver_employees WHERE name = 'Alice'"))

In [ ]:
# Time-Travel read: inspect version 0 (before update)
import os
table_path = os.path.join(os.getenv("WAREHOUSE_DIR", "/workspace/warehouse"), "dbo", "silver_employees")
print("Time-travel version 0 (original salary):")
display(conn.sql(f"SELECT name, salary FROM delta_scan('{table_path}', version => 0) WHERE name = 'Alice'"))